# Example: Propagating SIM Estimation Uncertainty into a Portfolio
This example replaces ad hoc independent parameter noise with bootstrap distributions estimated from the course dataset. We construct a fixed minimum-variance portfolio, then measure how its risk and weights change across plausible SIM calibrations.

> __Learning Objectives:__
>
> * __Connect estimation to allocation:__ Feed per-asset bootstrap draws into the SIM covariance matrix.
> * __Keep units consistent:__ Construct covariance directly from annualized growth-rate observations without inserting an extra $\Delta t$ factor.
> * __Measure decision uncertainty:__ Summarize portfolio volatility, weight distance, and variance regret.
> * __Expose resampling-model risk:__ Compare empirical-residual and Gaussian parametric bootstrap propagation.

___


## Setup, Data, and Prerequisites
The asset universe is deliberately small enough to inspect but is estimated from the same frozen dataset used in L6a. Each asset is fitted against SPY with a deterministic, local bootstrap seed.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


In [ ]:
raw_dataset = MyTrainingMarketDataSet()["dataset"];
maximum_days = nrow(raw_dataset["AAPL"]);
dataset = Dict(ticker => frame for (ticker, frame) in raw_dataset if nrow(frame) == maximum_days);
all_tickers = sort(collect(keys(dataset)));
Δt = 1 / 252;
growth_rates = log_growth_matrix(dataset, all_tickers; Δt=Δt, risk_free_rate=0.0);

market = growth_rates[:, findfirst(==("SPY"), all_tickers)];
tickers = ["AAPL", "AMD", "JPM", "MSFT", "NVDA", "XOM"];
asset_returns = [growth_rates[:, findfirst(==(ticker), all_tickers)] for ticker in tickers];
μm = mean(market);
σm = std(market);
println("Universe: $(join(tickers, ", "))")


## Task 1: Estimate a Bootstrap Distribution for Every Asset
For each ticker we retain the joint $(\alpha,\beta,\sigma_{g,\varepsilon})$ draw from every bootstrap replicate. Keeping each row together matters: sampling the three marginals independently would discard their within-fit dependence.


In [ ]:
residual_results = [bootstrap_sim(market, asset_returns[i], tickers[i];
    n_bootstrap=750, seed=5660 + i, method=:residual) for i in eachindex(tickers)];
parametric_results = [bootstrap_sim(market, asset_returns[i], tickers[i];
    n_bootstrap=750, seed=5760 + i, method=:parametric) for i in eachindex(tickers)];

estimation_table = DataFrame(
    ticker=tickers,
    alpha=[result.point_estimate.α for result in residual_results],
    beta=[result.point_estimate.β for result in residual_results],
    residual_growth_std=[result.point_estimate.σ_ε for result in residual_results],
    R_squared=[result.point_estimate.r² for result in residual_results],
    beta_ci_width=[result.confidence_intervals.beta[2] - result.confidence_intervals.beta[1] for result in residual_results],
);
pretty_table(estimation_table; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 2: Build the Calibrated Portfolio
Under the course growth-rate convention,
$$
\boldsymbol\Sigma_g=\sigma_{g,m}^2\boldsymbol\beta\boldsymbol\beta^\top+\operatorname{diag}(\sigma_{g,\varepsilon,1}^2,\ldots,\sigma_{g,\varepsilon,N}^2).
$$
There is no extra $\Delta t$ in this matrix because both market and residual standard deviations were estimated from $g_t$ observations. If we instead formed a covariance of one-day log returns $r_t=\Delta t g_t$, the whole matrix would scale as $\boldsymbol\Sigma_r=\Delta t^2\boldsymbol\Sigma_g$.


In [ ]:
residual_propagation = propagate_sim_uncertainty(residual_results, μm, σm;
    n_scenarios=1500, seed=5901);
parametric_propagation = propagate_sim_uncertainty(parametric_results, μm, σm;
    n_scenarios=1500, seed=5902);

allocation = DataFrame(
    ticker=tickers,
    expected_growth=residual_propagation.point_expected_growth,
    minimum_variance_weight=residual_propagation.point_weights,
);
pretty_table(allocation; table_format=TextTableFormat(borders=text_table_borders__simple))


## Task 3: Score Portfolio-Level Uncertainty
The calibrated weights are held fixed in each scenario. We compare their variance with the scenario-specific minimum achievable variance. Their difference is variance regret; total variation distance between the two weight vectors is reported as allocation distance.


In [ ]:
function propagation_row(name, result)
    volatility = sqrt.(result.fixed_variance)
    return (method=name,
        volatility_q05=quantile(volatility, 0.05),
        volatility_median=median(volatility),
        volatility_q95=quantile(volatility, 0.95),
        regret_median=median(result.variance_regret),
        regret_q95=quantile(result.variance_regret, 0.95),
        weight_distance_median=median(result.weight_distance),
        weight_distance_q95=quantile(result.weight_distance, 0.95))
end;

scorecard = DataFrame([
    propagation_row("residual", residual_propagation),
    propagation_row("parametric", parametric_propagation),
]);
pretty_table(scorecard; table_format=TextTableFormat(borders=text_table_borders__simple))


In [ ]:
p1 = histogram(sqrt.(residual_propagation.fixed_variance); bins=40, normalize=:pdf,
    c=:navy, alpha=0.6, label="residual", xlabel="Portfolio growth-rate std", ylabel="Density");
histogram!(p1, sqrt.(parametric_propagation.fixed_variance); bins=40, normalize=:pdf,
    c=:orange, alpha=0.45, label="parametric");
point_volatility = sqrt(dot(residual_propagation.point_weights,
    residual_propagation.point_covariance * residual_propagation.point_weights));
vline!(p1, [point_volatility]; c=:red, lw=2, ls=:dash, label="point estimate");
p2 = histogram(residual_propagation.weight_distance; bins=40, normalize=:pdf,
    c=:purple, alpha=0.6, label="residual", xlabel="Weight distance", ylabel="Density");
histogram!(p2, parametric_propagation.weight_distance; bins=40, normalize=:pdf,
    c=:green4, alpha=0.45, label="parametric");
plot(p1, p2; layout=(1, 2), size=(1000, 400))


## Interpretation and Limits
A wide portfolio distribution can arise even when every individual beta interval looks modest, because optimization is a nonlinear transformation of all parameters. The gap between the residual and parametric scorecards is a direct model-risk diagnostic: it measures sensitivity to the fitted innovation distribution.

This is still estimation uncertainty, not a complete forward market scenario. Both bootstraps condition on the observed market path, treat asset fits independently, and resample innovations i.i.d. The L3a volatility-clustering diagnostic explains the resulting limit: neither method reproduces persistent regimes. L7b adds correlation, rebalancing, and transaction-cost stresses rather than interpreting these intervals as forecasts.


## Summary

> __Key Takeaways:__
>
> * __Bootstrap rows are joint parameter draws:__ Preserve $(\alpha,\beta,\sigma_{g,\varepsilon})$ dependence within each fitted asset.
> * __Uncertainty must reach the decision layer:__ Portfolio variance, weights, and regret can be materially less stable than individual coefficients suggest.
> * __Resampling choice is model risk:__ Residual and parametric propagation expose sensitivity to heavy-tailed one-day innovations.
> * __Do not confuse estimation intervals with forward forecasts:__ Regime persistence, factor changes, and cross-asset residual dependence require additional stress models.

___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. The unconstrained analytical minimum-variance weights are used to isolate estimation effects; production portfolios require explicit position, turnover, liquidity, and concentration constraints.
